<a href="https://colab.research.google.com/github/ditto-1/Cognitive-Computing/blob/main/Assignment_9a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install easyocr opencv-python deepface pandas torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 25.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 MB 47.3 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 43.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [deepface]


In [ ]:
import os
import re
import cv2
import easyocr
import pandas as pd
import kagglehub

class MedicineLabelReaderModule:
    def __init__(self):
        print("Initializing EasyOCR and loading Knowledge Base...")
        # Initialize EasyOCR reader (CPU mode; set gpu=True if running on a GPU environment)
        self.reader = easyocr.Reader(['en'], gpu=False)

        # Locate and load the medicine dataset CSV from kagglehub cache
        kb_path = kagglehub.dataset_download("ujjwalaggarwal402/medicine-dataset")
        csv_file = os.path.join(kb_path, "medicine_dataset.csv")

        if os.path.exists(csv_file):
            self.df_kb = pd.read_csv(csv_file)
            print("Knowledge base loaded successfully!")
        else:
            # Fallback direct cache path
            fallback_path = "/root/.cache/kagglehub/datasets/ujjwalaggarwal402/medicine-dataset/versions/1/medicine_dataset.csv"
            self.df_kb = pd.read_csv(fallback_path)
            print("Knowledge base loaded from fallback path!")

    def process_label(self, image_path):
        """Processes an image path through preprocessing, OCR, regex parsing, and DB lookup."""
        if not os.path.exists(image_path):
            return f"Error: Image path '{image_path}' not found."

        print(f"\n--- Processing Label Image: {image_path} ---")

        # 1. Read and Preprocess Image
        img = cv2.imread(image_path)
        if img is None:
            return "Error: Could not read the image file."

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        filtered = cv2.bilateralFilter(gray, 11, 17, 17) # Reduces noise while keeping edges sharp

        # 2. Extract Text using EasyOCR
        ocr_results = self.reader.readtext(filtered, detail=0)
        full_text = " ".join(ocr_results)
        print(f"Raw OCR Extracted Text: {full_text}")

        # 3. Regex Parsing for Structured Fields
        parsed_info = {
            "Detected Drug Name": "Unknown",
            "Dosage": "Unknown",
            "Expiry Date": "Unknown"
        }

        # Regex patterns for dosage and expiry
        dosage_match = re.search(r'\b\d+\s?(mg|ml|g|gm|tablets|capsules)\b', full_text, re.IGNORECASE)
        expiry_match = re.search(r'(exp[:\s]?\d{2}[/-]\d{2}[/-]?\d{2,4})', full_text, re.IGNORECASE)

        if dosage_match:
            parsed_info["Dosage"] = dosage_match.group(0)
        if expiry_match:
            parsed_info["Expiry Date"] = expiry_match.group(0)

        # Extract potential drug name (usually the first word or line)
        words = full_text.split()
        drug_name_query = ""
        if words:
            drug_name_query = words[0].upper()
            parsed_info["Detected Drug Name"] = drug_name_query

        print("\nParsed Label Details:")
        for k, v in parsed_info.items():
            print(f"  - {k}: {v}")

        # 4. Query the CSV Knowledge Base
        print("\nSearching Knowledge Base Database...")
        name_col = self.df_kb.columns[0] # Assumes first column contains medicine names
        matches = self.df_kb[self.df_kb[name_col].str.contains(drug_name_query, case=False, na=False)]

        if not matches.empty:
            print("✓ Match found in database!")
            return matches.iloc[0].to_dict()
        else:
            return f"No exact matching records found in database for '{drug_name_query}'."


In [ ]:
import glob
import os

# 1. Point to your downloaded dataset folder (from your earlier path)
val_dir = "/kaggle/input/fake-vs-real-medicine-datasets-images/dataset/val"

# 2. Automatically find the first available image file
sample_images = glob.glob(os.path.join(val_dir, "**", "*.jpg"), recursive=True)

if sample_images:
    sample_image_path = sample_images[0]
    print("Using sample image path:", sample_image_path)

    # 3. Test your module with this path
    result = module1.process_label(sample_image_path)
    print(result)
else:
    print("No images found in the specified directory.")

Using sample image path: /kaggle/input/fake-vs-real-medicine-datasets-images/dataset/val/Real/images5119.jpg

--- Processing Label Image: /kaggle/input/fake-vs-real-medicine-datasets-images/dataset/val/Real/images5119.jpg ---
Raw OCR Extracted Text: 

Parsed Label Details:
  - Detected Drug Name: Unknown
  - Dosage: Unknown
  - Expiry Date: Unknown

Searching Knowledge Base Database...
✓ Match found in database!
{'Name': 'Acetocillin', 'Category': 'Antidiabetic', 'Dosage Form': 'Cream', 'Strength': '938 mg', 'Manufacturer': 'Roche Holding AG', 'Indication': 'Virus', 'Classification': 'Over-the-Counter'}
